## Extract Unique News Categories

This cell loads the JSON file at `../scraping_system/news/latest-news.json`, reads all article categories, and builds a deduplicated list named `unique_categories`.

It supports different JSON layouts (`list` of items or a `dict` containing keys like `articles`, `news`, `items`, or `data`) and handles categories stored as either:
- `category` or `categories`
- a string (including comma-separated values) or a list

At the end, it prints the total number of unique categories found.

In [34]:
import json
from pathlib import Path

news_json_path = Path("..") / "scraping_system" / "news" / "latest-news.json"

with news_json_path.open("r", encoding="utf-8") as f:
    news_data = json.load(f)

if isinstance(news_data, list):
    news_items = news_data
elif isinstance(news_data, dict):
    for key in ("articles", "news", "items", "data"):
        if key in news_data and isinstance(news_data[key], list):
            news_items = news_data[key]
            break
    else:
        news_items = [news_data]
else:
    news_items = []

raw_categories = []
for item in news_items:
    if not isinstance(item, dict):
        continue

    for field in ("category", "categories"):
        if field not in item:
            continue

        value = item[field]
        if isinstance(value, list):
            raw_categories.extend(value)
        elif isinstance(value, str):
            raw_categories.extend(part.strip() for part in value.split(","))

unique_categories = sorted({
    c.strip() for c in raw_categories if isinstance(c, str) and c.strip()
})

print("There are {} unique categories.".format(len(unique_categories)))

There are 39 unique categories.


## Embedding-Based Category Normalization

We map source-specific category names into a small set of normalized labels.

Two supported strategies:
- Use an English model and translate French and Arabic category tokens to English before scoring.
- Optionally switch to a multilingual model (better native French/Arabic coverage, heavier download).

How it works:
- words with similar meaning are close in vector space
- each category is cleaned and split into candidate tokens
- French and Arabic tokens are translated to English equivalents
- best label is selected by similarity if it passes the threshold; otherwise `other`

In [35]:
import gensim.downloader as api

# Set to True to use a multilingual model (French-friendly, larger download).
USE_MULTILINGUAL_MODEL = False

model_name = (
    "fasttext-wiki-news-subwords-300"
    if USE_MULTILINGUAL_MODEL
    else "glove-wiki-gigaword-100"
)

print(f"Loading embedding model: {model_name}")
model = api.load(model_name)

Loading embedding model: glove-wiki-gigaword-100


## Step1: Build Category Mapping

The mapping logic is now split into small, maintainable steps:

1. Define canonical labels and aliases.
2. Configure French/Arabic translation dictionaries and thresholds.
3. Define normalization and candidate-generation helper functions.
4. Run similarity scoring and build `category_mapping`.

Outputs:
- `category_mapping`: dictionary of original category -> normalized label
- `out_of_vocab_categories`: categories with no usable embedding tokens

In [36]:
import re
import unicodedata

canonical_labels = [
    "sport",
    "technology",
    "politics",
    "economy",
    "health",
    "culture",
    "society",
    "education",
]

label_aliases = {
    "sport": [
        "sport", "sports", "football", "basketball", "soccer", "athlete", "match",
        "competition", "team"
    ],
    "technology": [
        "technology", "tech", "digital", "innovation", "software", "ai"
    ],
    "politics": [
        "politics", "political", "government", "diplomacy", "policy", "state",
        "presidency", "national", "international", "world", "news"
    ],
    "economy": [
        "economy", "economic", "business", "finance", "industry", "market", "trade",
        "energy", "bank"
    ],
    "health": [
        "health", "medical", "medicine", "hospital", "care", "wellness", "environment"
    ],
    "culture": [
        "culture", "cultural", "art", "arts", "media", "heritage"
    ],
    "society": [
        "society", "social", "community", "public", "local"
    ],
    "education": [
        "education", "school", "university", "learning", "student"
    ],
}

print("Step ready: labels and aliases configured")

Step ready: labels and aliases configured


## Step 2: Configure Translation Dictionaries

This cell defines French -> English and Arabic -> English token dictionaries, plus shared configuration like the Arabic diacritics regex and classification threshold.

In [37]:
french_to_english = {
    "actualite": "news",
    "nationale": "national",
    "national": "national",
    "monde": "world",
    "arabe": "arab",
    "politique": "politics",
    "gouvernement": "government",
    "presidence": "presidency",
    "societe": "society",
    "sport": "sport",
    "sports": "sports",
    "sante": "health",
    "technologie": "technology",
    "numerique": "digital",
    "education": "education",
    "ecole": "school",
    "universite": "university",
    "economie": "economy",
    "economique": "economic",
    "banque": "bank",
    "finances": "finance",
    "commerce": "trade",
    "service": "service",
    "industrie": "industry",
    "energie": "energy",
    "mines": "mines",
    "culture": "culture",
    "culturel": "cultural",
    "environnement": "environment",
    "developpement": "development",
    "algerie": "algeria",
}

arabic_to_english = {
    "اخبار": "news",
    "خبر": "news",
    "العالم": "world",
    "عالم": "world",
    "الجزائر": "algeria",
    "جزائر": "algeria",
    "الرياضة": "sport",
    "رياضة": "sport",
    "سياسة": "politics",
    "سياسي": "political",
    "اقتصاد": "economy",
    "اقتصادي": "economic",
    "تكنولوجيا": "technology",
    "تقنية": "technology",
    "ثقافة": "culture",
    "ثقافي": "cultural",
    "الوطني": "national",
    "وطني": "national",
    "محلي": "local",
    "منوعات": "variety",
    "الحدث": "event",
    "قناة": "channel",
    "النهار": "ennahar",
    "النسخة": "edition",
    "المصورة": "illustrated",
}

arabic_diacritics_pattern = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u0640]")
threshold = 0.35

print("Step ready: translation dictionaries and thresholds configured")

Step ready: translation dictionaries and thresholds configured


## Step 3: Build Normalization and Translation Helpers

These helpers clean category text, normalize Arabic/French forms, and generate robust English candidate tokens for embedding similarity.

In [38]:
def strip_accents(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in normalized if not unicodedata.combining(ch))


def normalize_arabic(text: str) -> str:
    return arabic_diacritics_pattern.sub("", text)


def translate_arabic_token(token: str) -> str:
    if token in arabic_to_english:
        return arabic_to_english[token]

    if token.startswith("ال") and len(token) > 2:
        without_prefix = token[2:]
        if without_prefix in arabic_to_english:
            return arabic_to_english[without_prefix]

    return token


def translate_to_english_tokens(text: str) -> str:
    lowered = text.lower()
    ascii_text = strip_accents(lowered)
    arabic_text = normalize_arabic(lowered)

    latin_tokens = re.findall(r"[a-z]+", ascii_text)
    arabic_tokens = re.findall(r"[\u0600-\u06FF]+", arabic_text)

    translated_latin = [french_to_english.get(token, token) for token in latin_tokens]
    translated_arabic = [translate_arabic_token(token) for token in arabic_tokens]

    translated = [token for token in (translated_latin + translated_arabic) if token]
    return " ".join(translated).strip()


def category_candidates(category_name: str):
    cleaned = category_name.strip().lower()
    normalized_latin = strip_accents(cleaned)
    normalized_arabic = normalize_arabic(cleaned)

    split_ready = (
        normalized_latin
        .replace("&", " and ")
        .replace("|", "/")
        .replace("-", " ")
        .replace(",", "/")
        .replace("،", "/")
        .replace("؛", "/")
    )

    parts = [p.strip() for p in split_ready.split("/") if p.strip()]
    candidates = set(parts + [normalized_latin, normalized_arabic])

    for part in parts:
        candidates.update(token for token in part.split() if token)

        translated_part = translate_to_english_tokens(part)
        if translated_part:
            candidates.add(translated_part)
            candidates.update(translated_part.split())

    translated_full = translate_to_english_tokens(cleaned)
    if translated_full:
        candidates.add(translated_full)
        candidates.update(translated_full.split())

    return sorted(candidates)

print("Step ready: normalization and translation helpers loaded")

Step ready: normalization and translation helpers loaded


## Step 4: Build the Final Mapping

This cell applies the helpers and label aliases to classify each source category.

Result variables:
- `category_mapping`
- `out_of_vocab_categories`

In [39]:
available_label_aliases = {
    label: [alias for alias in aliases if alias in model.key_to_index]
    for label, aliases in label_aliases.items()
}

category_mapping = {}
out_of_vocab_categories = []

for category in unique_categories:
    category_terms = [
        term for term in category_candidates(category)
        if term in model.key_to_index
    ]

    if not category_terms:
        category_mapping[category] = "other"
        out_of_vocab_categories.append(category)
        continue

    best_label = "other"
    best_score = -1.0

    for label in canonical_labels:
        label_terms = available_label_aliases.get(label, [])
        if not label_terms:
            continue

        score = max(
            model.similarity(category_term, label_term)
            for category_term in category_terms
            for label_term in label_terms
        )

        if score > best_score:
            best_score = score
            best_label = label

    category_mapping[category] = best_label if best_score >= threshold else "other"

print(f"Mapped {len(category_mapping)} categories")
print(f"Out-of-vocabulary categories: {len(out_of_vocab_categories)}")
print("Sample mapping:")
for source_category, mapped_label in list(category_mapping.items())[:20]:
    print(f"- {source_category} -> {mapped_label}")

Mapped 39 categories
Out-of-vocabulary categories: 0
Sample mapping:
- Algérie / Actualité Nationale -> politics
- Algérie / Développement -> health
- Algérie / Santé et Environnement -> health
- Algérie / Société -> society
- Algérie / Éducation et Technologie -> technology
- Basketball -> sport
- Culture / Arts -> culture
- Economie / Banque et Finances -> economy
- Economie / Commerce et Service -> economy
- Economie / Industrie -> economy
- Energie et mines -> economy
- Football -> sport
- L'Actualité en temps réel -> politics
- Monde / Monde Arabe -> politics
- National -> politics
- National Team -> sport
- Politique -> politics
- Présidence News -> politics
- Sports / Sport Collectif -> sport
- politics -> politics


## Step 5: Reindex News and Rewrite Main JSON

This step prepares article-level category pairs and applies mapped categories back to the main news file.

What it saves:
- `news-category-index.json`: list of `news_index` with old/new categories
- `category-encoding-pairs.json`: encoded old/new category pairs
- `latest-news.json`: rewritten with mapped categories and per-article category pair metadata
- `latest-news.backup.json`: one-time backup of the original main file

In [40]:
def extract_old_categories(article: dict):
    values = []

    for field in ("category", "categories"):
        if field not in article:
            continue

        value = article[field]
        if isinstance(value, list):
            values.extend(value)
        elif isinstance(value, str):
            values.extend(part.strip() for part in value.split(","))

    ordered_unique = []
    seen = set()

    for value in values:
        if not isinstance(value, str):
            continue

        cleaned = value.strip()
        if not cleaned or cleaned in seen:
            continue

        seen.add(cleaned)
        ordered_unique.append(cleaned)

    return ordered_unique


old_category_codes = {
    category: index
    for index, category in enumerate(sorted(category_mapping.keys()), start=1)
}

new_category_codes = {
    category: index
    for index, category in enumerate(sorted(set(category_mapping.values())), start=1)
}

category_encoding_pairs = []
for old_category in sorted(category_mapping.keys()):
    new_category = category_mapping[old_category]
    category_encoding_pairs.append({
        "old_category": old_category,
        "old_category_code": old_category_codes[old_category],
        "new_category": new_category,
        "new_category_code": new_category_codes[new_category],
    })

news_category_index = []
updated_news_items = []

for news_index, article in enumerate(news_items):
    if not isinstance(article, dict):
        continue

    old_categories = extract_old_categories(article)
    if not old_categories:
        old_categories = ["uncategorized"]

    mapped_categories = sorted({
        category_mapping.get(old_category, "other")
        for old_category in old_categories
    })

    pair_rows = []
    for old_category in old_categories:
        mapped_category = category_mapping.get(old_category, "other")
        pair_rows.append({
            "old_category": old_category,
            "old_category_code": old_category_codes.get(old_category),
            "new_category": mapped_category,
            "new_category_code": new_category_codes[mapped_category],
        })

    news_category_index.append({
        "news_index": news_index,
        "old_categories": old_categories,
        "new_categories": mapped_categories,
        "category_pairs": pair_rows,
    })

    updated_article = dict(article)
    updated_article["old_categories"] = old_categories
    updated_article["new_categories"] = mapped_categories
    updated_article["category_encoding_pairs"] = pair_rows
    updated_article["category"] = mapped_categories[0] if len(mapped_categories) == 1 else mapped_categories
    updated_news_items.append(updated_article)


def merge_updated_items(original_data, updated_items):
    if isinstance(original_data, list):
        return updated_items

    if isinstance(original_data, dict):
        updated = dict(original_data)
        for key in ("articles", "news", "items", "data"):
            if key in updated and isinstance(updated[key], list):
                updated[key] = updated_items
                return updated

    return updated_items


updated_news_data = merge_updated_items(news_data, updated_news_items)

news_index_path = news_json_path.parent / "news-category-index.json"
encoding_pairs_path = news_json_path.parent / "category-encoding-pairs.json"
backup_path = news_json_path.parent / "latest-news.backup.json"

with news_index_path.open("w", encoding="utf-8") as f:
    json.dump(news_category_index, f, ensure_ascii=False, indent=2)

with encoding_pairs_path.open("w", encoding="utf-8") as f:
    json.dump(category_encoding_pairs, f, ensure_ascii=False, indent=2)

if not backup_path.exists():
    with backup_path.open("w", encoding="utf-8") as f:
        json.dump(news_data, f, ensure_ascii=False, indent=2)

with news_json_path.open("w", encoding="utf-8") as f:
    json.dump(updated_news_data, f, ensure_ascii=False, indent=2)

print(f"Updated main news JSON: {news_json_path.resolve()}")
print(f"Created backup (if missing): {backup_path.resolve()}")
print(f"Saved news index list: {news_index_path.resolve()}")
print(f"Saved category encoding pairs: {encoding_pairs_path.resolve()}")
print(f"Indexed news rows: {len(news_category_index)}")

Updated main news JSON: C:\Work\current\news platform\scraping_system\news\latest-news.json
Created backup (if missing): C:\Work\current\news platform\scraping_system\news\latest-news.backup.json
Saved news index list: C:\Work\current\news platform\scraping_system\news\news-category-index.json
Saved category encoding pairs: C:\Work\current\news platform\scraping_system\news\category-encoding-pairs.json
Indexed news rows: 260


## Export Mapping to JSON

This cell saves the mapping for reuse in our scraping or website pipeline.

Saved file includes:
- timestamp (`created_at`)
- total source categories
- OOV category list
- final `category_mapping`

Output path:
- `../scraping_system/news/category-mapping.json`

In [41]:
from datetime import datetime, timezone

mapping_output = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "total_unique_categories": len(unique_categories),
    "mapped_categories": len(category_mapping),
    "out_of_vocabulary_categories": out_of_vocab_categories,
    "category_mapping": category_mapping,
}

mapping_output_path = Path("..") / "scraping_system" / "news" / "category-mapping.json"

with mapping_output_path.open("w", encoding="utf-8") as f:
    json.dump(mapping_output, f, ensure_ascii=False, indent=2)

print(f"Saved mapping to: {mapping_output_path.resolve()}")

Saved mapping to: C:\Work\current\news platform\scraping_system\news\category-mapping.json
